<a href="https://colab.research.google.com/github/moodyastra/Quickstart_Pymol/blob/main/Excercise2_use_AlphaFold3_type_predictors_to_predict_enzyme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting an Protein–ATP Complex with AlphaFold 3 in Google Colab

**Author:** Linna An edited based on [ColabFold-Boltz](https://github.com/sokrypton/ColabFold)

This notebook demonstrates how to use **Google Colab** to to predict the 3‑D structure of protein bound to the ligand **ATP** using ColabFold, a open-sourced user-friendly scripts which incorporated all curently commonly seen prediction tools, such as **[AlphaFold 2 (AF2)](https://github.com/google-deepmind/alphafold)**, [Boltz](https://github.com/jwohlwend/boltz) (a MIT-recreated open sourced AF3). Based on the predicted structure, we can come up with hypothesis of if this binding could be real or not. It covers:

1. Intro to Colab, Jupyter, MSAs & AlphaFold-type of predictors.
2. Preparing an input FASTA that specifies both protein & ligand
3. Generating a Multiple Sequence Alignment (MSA) with MMseqs2
4. Running AF3 inference (with pretrained weights)
5. Visualising the resulting complex in‑notebook


# This is a work-in-progress notebook for [Boltz](https://github.com/jwohlwend/boltz)

⚠️ **Warning to Users:**
- **Alpha Version:** This notebook is currently under active development and is considered a beta version.
- **Usage at Your Own Risk:** Use this notebook at your own discretion and risk.

## 1. What are Google Colab & Jupyter notebooks?
- **Jupyter notebook**: an interactive document that combines live code, formatted text, math, and plots. Each *cell (or code block)* can be executed individually.
- **Google Colab**:
  * Google’s free cloud hosting of Jupyter notebooks.
  * No local setup (aka you do not need to borther with learning how to install any software);
  * free CPUs/GPUs (by copying this colab to your own drive, you request a cpu/gpu memory resource from google);
  * notebooks auto‑save to Google Drive.

📌 **Usage tips**  
- Copy to your own google drive using **File -> Save as a copy in drive**
- Run a cell with **Shift + Enter** or click the ▶ icon.  
- Enable a GPU under **Runtime ▸ Change runtime type**, use T4/GPU. AF3-type of predictors needs to be ran on GPU, so change to a GPU.
- Your runtime is temporary—save important files to Drive before disconnects.
- use the down arraw to see the source code

## 2 Core concepts
### 2.1 Multiple Sequence Alignment (MSA)
- [what is a MSA?](https://www.youtube.com/watch?v=TZaA_-4j19w)
- Aligns homologous sequences column‑wise; reveals conserved, functionally important residues used by AlphaFold.

### 2.2 AlphaFold 3 (AF3)
- [A 5 min peek of AlphaFold](https://www.youtube.com/watch?v=7q8Uw3rmXyE)
- [A comprehensive 1-hour discussion of AlphaFold3 by Sergey O](https://www.youtube.com/watch?v=qjFgthkKxcA)
- Use Next‑gen model (here use AlphaFold3) that predicts full complexes: proteins + DNA/RNA + **small molecule ligands**.

### 2.3 ATP & ATPases
- ATP (adenosine triphosphate) is the cell’s energy currency. **ATPases** are enzymes that hydrolyse ATP, using the released energy for work.
- Designing ATP-binding or ATP-hydrolyzing protein will allow we generate fuels for protein nanomachine (another 2016 Nobel prize-winning hot field).

In [ ]:
#@title Install dependencies
%%time
import os
import sys

if not os.path.isfile("COLABFOLD_READY"):
  print("installing colabfold...")
  result = os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
  if result != 0:
    raise RuntimeError("ColabFold installation failed")
  if os.environ.get('TPU_NAME', False) != False:
    os.system("pip uninstall -y jax jaxlib")
    os.system("pip install --no-warn-conflicts --upgrade dm-haiku==0.0.10 'jax[cuda12_pip]'==0.3.25 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")
  os.system("ln -sf /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
  os.system("ln -sf /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
  os.system("touch COLABFOLD_READY")

if not os.path.isfile("BOLZ_READY_V2"):
  print("installing boltz...")
  # Use the compatibility-maintained package so current Colab Python versions
  # can install a recent dm-tree wheel instead of trying to build dm-tree 0.1.8.
  os.system(f"{sys.executable} -m pip uninstall -y -q boltz boltz-community")
  result = os.system(f"{sys.executable} -m pip install -q -U boltz-community py3Dmol")
  if result != 0:
    raise RuntimeError("Boltz installation failed")

  os.makedirs("weights", exist_ok=True)
  if os.system("boltz --help > /dev/null 2>&1") != 0:
    raise RuntimeError("Boltz installed but the command is not working")
  os.system("touch BOLZ_READY_V2")

# Old Code:
# #@title Install dependencies
# %%time
# import os
# if not os.path.isfile("COLABFOLD_READY"):
# print("installing colabfold...")
# os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
# if os.environ.get('TPU_NAME', False) != False:
#   os.system("pip uninstall -y jax jaxlib")
#   os.system("pip install --no-warn-conflicts --upgrade dm-haiku==0.0.10 'jax[cuda12_pip]'==0.3.25 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")
#   os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
#   os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
#   os.system("touch COLABFOLD_READY")
#
# if not os.path.isfile("BOLZ_READY"):
#    os.system("apt-get install -y aria2")
#    os.system("pip install -q --no-warn-conflicts boltz")
#    os.system("mkdir weights")
#    os.system("aria2c -d weights -x8 -s8 https://colabfold.steineggerlab.workers.dev/boltz1.ckpt")
#    os.system("aria2c -d weights -x8 -s8 https://colabfold.steineggerlab.workers.dev/ccd.pkl")
#    os.system("touch BOLZ_READY")


installing colabfold...
installing boltz...
CPU times: user 5.37 ms, sys: 1.1 ms, total: 6.48 ms
Wall time: 1min 2s


### Compatibility fixes made
- Added explicit install-failure checks for ColabFold instead of continuing after a failed `pip` command.
- Made ColabFold symlink creation repeat-safe with `ln -sf` instead of `ln -s`.
- Replaced the failing `boltz` install with `boltz-community`, using the active Python interpreter and a fresh `BOLZ_READY_V2` marker.
- Removed the old `aria2`/manual checkpoint + CCD downloads; Boltz now manages files through `--cache weights`.
- Added Boltz CLI verification and safe `weights` directory creation.
- Removed the forced nightly PyTorch/CUDA 11.8 reinstall; the notebook now checks Colab's existing GPU-enabled PyTorch runtime instead.
- Added explicit Boltz/ColabFold CLI and CUDA verification before running predictions.
- Changed the protein MSA reference from a relative job-folder path to an absolute `.a3m` path, preventing the duplicated `jobname/jobname/...` path error.
- Replaced the shell MSA command with `subprocess.run(..., check=True)` and verify every expected `.a3m` file exists before Boltz starts.
- Fixed the Boltz prediction command (including the stray trailing backtick), use the generated FASTA path directly, explicitly select `boltz1`, add `--no_kernels` for Colab compatibility, and `--override` to replace partial failed runs.
- Run Boltz with `subprocess.run(..., check=True)` so prediction failures stop at the real error.
- Replaced the hard-coded prediction-folder `ls` with output-directory discovery that supports both current and older Boltz layouts.
- Simplified visualization setup to `py3Dmol`; removed unnecessary system PyMOL and `icecream` installs.
- Replaced the single hard-coded CIF path + `assert` with current/legacy path checks plus a recursive fallback and an informative `FileNotFoundError`.
- Kept the original visualization behavior while making the cell self-contained and reading the located CIF into `cif_text`.


In [ ]:
#@title Check PyTorch runtime
# A majority of the updated code was generated with AI assistance because the original program repeatedly produced errors even after 5 hours of trouble shooting (with my limited knowledge).

import torch
print("torch:", torch.__version__)
print("CUDA in torch:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
print("torch.library.custom_op available:", hasattr(torch.library, "custom_op"))
if not torch.cuda.is_available():
  print("Warning: switch Colab to a GPU runtime before running Boltz.")

# Old Code:
# #@title Reinstall torch to remove a bug from public version (LA)
# # restart will be needed after reinstallation
# # REMOVE every old build first
# !pip uninstall -y torch torchvision torchaudio
#
# # ---- GPU, CUDA 11.8 example ----
# !pip install --pre torch torchvision torchaudio \
#      --index-url https://download.pytorch.org/whl/nightly/cu118
# # ---- GPU, CUDA 12.1 example ----
# # pip install --pre torch torchvision torchaudio \
# #      --index-url https://download.pytorch.org/whl/nightly/cu121
# # ---- CPU fallback ----
# # pip install --pre torch torchvision torchaudio \
# #      --index-url https://download.pytorch.org/whl/nightly/cpu


torch: 2.11.0+cu128
CUDA in torch: 12.8
GPU available: True
torch.library.custom_op available: True


In [ ]:
#@title Input protein sequence(s), then hit `Runtime` -> `Run all`
from google.colab import files
import os
import re
import hashlib
import random
import requests
from string import ascii_uppercase

# Function to add a hash to the jobname
def add_hash(x, y):
    return x + "_" + hashlib.sha1(y.encode()).hexdigest()[:5]

# User inputs
query_sequence = 'SPRLEENTDRIIEAIRAGKIPRSVLEEWRRRSREDSDAMTPEERRALEEAHEAGEVEITGETFQVPEVPWLYYGRGVYEKFGRRLTGTLSLTGEPIDTPLRESREVSRRIDAALDEVEA'  #@param {type:"string"}
#@markdown  - Use `:` to specify inter-protein chainbreaks for **modeling complexes** (supports homo- and hetro-oligomers). For example **PI...SK:PI...SK** for a homodimer
ligand_input = 'C1=NC(=C2C(=N1)N(C=N2)[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O)N'  #@param {type:"string"}
#@markdown  - Use `:` to specify multiple ligands as smile strings
ligand_input_ccd = 'ATP'  #@param {type:"string"}
#@markdown - Use `:` to specify multiple ligands as CCD codes (three-letter codes)
ligand_input_common_name = ''  #@param {type:"string"}
#@markdown - Use `:` to specify multiple ligands with their common name (e.g. Aspirin; SMILES fetched from [PubChem](https://pubchem.ncbi.nlm.nih.gov) API)
dna_input = ''  #@param {type:"string"}
#@markdown - Use `:` to specify multiple DNA sequences
jobname = 'test'  #@param {type:"string"}

# Clean up the query sequence and jobname
query_sequence = "".join(query_sequence.split())
ligand_input = "".join(ligand_input.split())
ligand_input_ccd = "".join(ligand_input_ccd.split())
ligand_input_common_name = "".join(ligand_input_common_name.split())
dna_input = "".join(dna_input.split())
basejobname = "".join(jobname.split())
basejobname = re.sub(r'\W+', '', basejobname)
jobname = add_hash(basejobname, query_sequence)

# Check if a directory with jobname exists
def check(folder):
    return not os.path.exists(folder)

if not check(jobname):
    n = 0
    while not check(f"{jobname}_{n}"):
        n += 1
    jobname = f"{jobname}_{n}"

# Make directory to save results
os.makedirs(jobname, exist_ok=True)

from string import ascii_uppercase

# Split sequences on chain breaks
protein_sequences = query_sequence.strip().split(':') if query_sequence.strip() else []
ligand_sequences = ligand_input.strip().split(':') if ligand_input.strip() else []
ligand_sequences_ccd = ligand_input_ccd.strip().split(':') if ligand_input_ccd.strip() else []
ligand_sequences_common_name = ligand_input_common_name.strip().split(':') if ligand_input_common_name.strip() else []
dna_sequences = dna_input.strip().split(':') if dna_input.strip() else []

def get_smiles(compound_name):
    autocomplete_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/autocomplete/compound/{compound_name}/json?limit=1"
    autocomplete_response = requests.get(autocomplete_url)
    if autocomplete_response.status_code != 200:
        return None

    autocomplete_data = autocomplete_response.json()
    if autocomplete_data.get("status", {}).get("code") != 0 or autocomplete_data.get("total", 0) == 0:
        return None

    suggested_compound = autocomplete_data.get("dictionary_terms", {}).get("compound", [])
    if not suggested_compound:
        return None
    suggested_compound_name = suggested_compound[0]

    smiles_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{suggested_compound_name}/property/CanonicalSMILES/JSON"
    smiles_response = requests.get(smiles_url)
    if smiles_response.status_code != 200:
        return None

    smiles_data = smiles_response.json()
    properties = smiles_data.get("PropertyTable", {}).get("Properties", [])
    if len(properties) == 0:
        return None

    return properties[0].get("CanonicalSMILES")

smiles_cache = {}
for name in ligand_sequences_common_name:
    if name not in smiles_cache:
        smiles_cache[name] = get_smiles(name)
        if smiles_cache[name] is not None:
          print(f"Mapped compound {name} to {smiles_cache[name]}")

    if smiles_cache[name] is not None:
        ligand_sequences.append(smiles_cache[name])

# Initialize chain labels starting from 'A'
chain_labels = iter(ascii_uppercase)

fasta_entries = []
csv_entries = []
chain_label_to_seq_id = {}
seq_to_seq_id = {}
seq_id_counter = 0  # Counter for unique sequences

# Process protein sequences
for seq in protein_sequences:
    seq = seq.strip()
    if not seq:
        continue  # Skip empty sequences
    chain_label = next(chain_labels)
    # Check if sequence has been seen before
    if seq in seq_to_seq_id:
        seq_id = seq_to_seq_id[seq]
    else:
        seq_id = f"{jobname}_{seq_id_counter}"
        seq_to_seq_id[seq] = seq_id
        seq_id_counter += 1
        # For CSV file (for ColabFold), add only unique sequences
        csv_entries.append((seq_id, seq))
    chain_label_to_seq_id[chain_label] = seq_id
    # For FASTA file
    msa_path = os.path.abspath(os.path.join(jobname, f"{seq_id}.a3m"))  # unambiguous path for Boltz
    # Old Code:
    # msa_path = os.path.join(jobname, f"{seq_id}.a3m")
    header = f">{chain_label}|protein|{msa_path}"
    sequence = seq
    fasta_entries.append((header, sequence))

# Process ligand sequences (assumed to be SMILES strings)
for lig in ligand_sequences:
    lig = lig.strip()
    if not lig:
        continue  # Skip empty ligands
    chain_label = next(chain_labels)
    lig_type = 'smiles'
    header = f">{chain_label}|{lig_type}"
    sequence = lig
    fasta_entries.append((header, sequence))

# Process DNA sequences (NO MSA is generated)
for seq in dna_sequences:
    seq = seq.strip()
    if not seq:
        continue  # Skip empty sequences
    chain_label = next(chain_labels)
    lig_type = 'DNA'
    header = f">{chain_label}|{lig_type}"
    sequence = seq
    fasta_entries.append((header, sequence))

# Process ligand sequences (CCD codes)
for lig in ligand_sequences_ccd:
    lig = lig.strip()
    if not lig:
        continue  # Skip empty ligands
    chain_label = next(chain_labels)
    lig_type = 'ccd'
    header = f">{chain_label}|{lig_type}"
    sequence = lig.upper()  # Ensure CCD codes are uppercase
    fasta_entries.append((header, sequence))

# Write the CSV file for ColabFold
queries_path = os.path.join(jobname, f"{jobname}.csv")
with open(queries_path, "w") as text_file:
    text_file.write("id,sequence\n")
    for seq_id, seq in csv_entries:
        text_file.write(f"{seq_id},{seq}\n")

# Write the FASTA file
queries_fasta = os.path.join(jobname, f"{jobname}.fasta")
with open(queries_fasta, 'w') as f:
    for header, sequence in fasta_entries:
        f.write(f"{header}\n{sequence}\n")

# Optionally, print the output for verification
#print(f"Generated FASTA file '{queries_fasta}':\n")
#for header, sequence in fasta_entries:
#    print(f"{header}\n{sequence}\n")


In [ ]:
#@title Verify core installations
import shutil
import torch

print("Boltz CLI:", shutil.which("boltz"))
print("ColabFold CLI:", shutil.which("colabfold_batch"))
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if shutil.which("boltz") is None:
  raise RuntimeError("Boltz command was not installed correctly")
if shutil.which("colabfold_batch") is None:
  raise RuntimeError("ColabFold command was not installed correctly")

# Old Code:
# # check the installations
# import torch, torchvision, platform, sys, subprocess, os
# print("torch :", torch.__version__) # should be higher than 2.4.x
# print("torchvision:", torchvision.__version__)
# print("CUDA in torch:", torch.version.cuda)
# print("torch is at :", torch.__file__)
# print("torchvision is at :", torchvision.__file__)
# import torch.library              # this should have custom_op now
# print(hasattr(torch.library,"custom_op"))  # should give True


Boltz CLI: /usr/local/bin/boltz
ColabFold CLI: /usr/local/bin/colabfold_batch
PyTorch: 2.11.0+cu128
CUDA available: True


In [ ]:
#@title Generate MSA with ColabFold
import os
import subprocess

subprocess.run(
    ["colabfold_batch", queries_path, jobname, "--msa-only"],
    check=True,
)

# Boltz reads each custom MSA relative to the FASTA file's directory.
expected_msas = [os.path.join(jobname, f"{seq_id}.a3m") for seq_id, _ in csv_entries]
missing_msas = [path for path in expected_msas if not os.path.isfile(path)]
if missing_msas:
    raise FileNotFoundError(
        "ColabFold finished, but these expected MSA files were not created: "
        + ", ".join(missing_msas)
    )

print("MSA generation complete:")
for path in expected_msas:
    print(" -", path) #ACTUAL MSA

# Old Code:
# #@title Generate MSA with ColabFold
# !colabfold_batch "{queries_path}" "{jobname}" --msa-only


MSA generation complete:
 - test_b1c13/test_b1c13_0.a3m


In [ ]:
!ls # see what are the files generated, you should be able to see your job directories

alphafold      colabfold	sample_data  weights
BOLZ_READY_V2  COLABFOLD_READY	test_b1c13


In [ ]:
queries_path # this should be the msa (Are csvs MSAs? Or only ".a3m" files)

'test_b1c13/test_b1c13.csv'

In [ ]:
!cat $queries_path # check the msa content, if you used the provided sequence, then you should only see 2 lines. Think about it, does it fit the expected MSA format you read above?
# No, it doesn't seem to fit the "expected" MSA format I read above as the csv contains a sequence identifier and the query sequence, not the generated MSA.

id,sequence
test_b1c13_0,SPRLEENTDRIIEAIRAGKIPRSVLEEWRRRSREDSDAMTPEERRALEEAHEAGEVEITGETFQVPEVPWLYYGRGVYEKFGRRLTGTLSLTGEPIDTPLRESREVSRRIDAALDEVEA


In [ ]:
jobname

'test_b1c13'

In [ ]:
jobname # check the running job

'test_b1c13'

In [ ]:
!ls # check the files in the folder

alphafold      colabfold	sample_data  weights
BOLZ_READY_V2  COLABFOLD_READY	test_b1c13


In [ ]:
#@title Predict structure using Boltz
# --no_kernels avoids optional CUDA-kernel compatibility problems on Colab GPUs.
# --override prevents a previous failed/partial run from being reused.
import subprocess

boltz_cmd = [
    "boltz", "predict", queries_fasta,
    "--out_dir", jobname,
    "--cache", "weights",
    "--model", "boltz1",
    "--no_kernels",
    "--override",
]
print("Running:", " ".join(boltz_cmd))
subprocess.run(boltz_cmd, check=True)

# Old Code:
# #@title Predict structure using boltz
# !boltz predict --out_dir "{jobname}" "{jobname}/{jobname}.fasta" --cache weights`


Running: boltz predict test_b1c13/test_b1c13.fasta --out_dir test_b1c13 --cache weights --model boltz1 --no_kernels --override


CompletedProcess(args=['boltz', 'predict', 'test_b1c13/test_b1c13.fasta', '--out_dir', 'test_b1c13', '--cache', 'weights', '--model', 'boltz1', '--no_kernels', '--override'], returncode=0)

In [ ]:
#@title Check prediction output
import os
import glob

candidate_dirs = [
    os.path.join(jobname, "predictions", jobname),
    os.path.join(jobname, f"boltz_results_{jobname}", "predictions", jobname),
]
pred_dir = next((p for p in candidate_dirs if os.path.isdir(p)), None)

if pred_dir is None:
    matches = glob.glob(os.path.join(jobname, "**", "predictions", jobname), recursive=True)
    pred_dir = matches[0] if matches else None

if pred_dir is None:
    raise FileNotFoundError(
        "Boltz finished without a prediction directory. Check the Boltz cell output above."
    )

print("Prediction directory:", pred_dir)
for name in sorted(os.listdir(pred_dir)):
    print(name)

# Old Code:
# !ls $jobname/boltz_results_$jobname/predictions/$jobname/ # check the content of the job folder, the `cif` file is the prediction results.


Prediction directory: test_b1c13/boltz_results_test_b1c13/predictions/test_b1c13
confidence_test_b1c13_model_0.json
pae_test_b1c13_model_0.npz
pde_test_b1c13_model_0.npz
plddt_test_b1c13_model_0.npz
test_b1c13_model_0.cif


In [ ]:
#@title Install visualization dependency
!pip -q install -U py3Dmol

# Old Code:
# #@title Visialize the results using Pymol
# # install pymol
# !apt-get update -y        # -qq keeps the log short
# !apt-get install -y pymol # open-source PyMOL 2.x
# !pip -q install py3Dmol
# !pip install icecream # this is a package commonly used for debugging


In [ ]:
#@title Visualize the prediction
import os
import glob
import py3Dmol

candidate_cifs = [
    os.path.join(jobname, "predictions", jobname, f"{jobname}_model_0.cif"),
    os.path.join(
        jobname,
        f"boltz_results_{jobname}",
        "predictions",
        jobname,
        f"{jobname}_model_0.cif",
    ),
]
pred_results = next((p for p in candidate_cifs if os.path.isfile(p)), None)

if pred_results is None:
    matches = glob.glob(os.path.join(jobname, "**", "*_model_0.cif"), recursive=True)
    if not matches:
        raise FileNotFoundError(
            "No Boltz prediction CIF was found. Run the Boltz prediction cell successfully first."
        )
    pred_results = matches[0]

print("Using prediction:", pred_results)
with open(pred_results, "r") as f:
    cif_text = f.read()

view = py3Dmol.view(width=500, height=450)
view.addModel(cif_text, "cif")
view.setStyle({"cartoon": {}})
view.addStyle(
    {"chain": "B", "hetflag": True},
    {"stick": {"radius": 0.25, "colorscheme": "element"}},
)
view.zoomTo()
view.show()

# Old Code:
# import io
# import py3Dmol
# from IPython.display import display
#
# pred_results = os.path.join(jobname, f"boltz_results_{jobname}", "predictions", jobname, f"{jobname}_model_0.cif")
# assert os.path.isfile(pred_results) # verify the result exists
#
# with open(pred_results,'r') as f:
#    pdb_text = f.read()
#    print(len(pdb_text))
# # Push that PDB string into py3Dmol and show a cartoon
#    view = py3Dmol.view(width=500, height=450)
#    view.addModel(pdb_text, 'cif')      # just pass the string
#    view.setStyle({'cartoon': {}})
#    view.addStyle({'chain': 'B', 'hetflag': True},  # selection
#                  {'stick': {'radius': 0.25, 'colorscheme': 'element'}})
#    view.zoomTo()
#    view.show()


Using prediction: test_b1c13/boltz_results_test_b1c13/predictions/test_b1c13/test_b1c13_model_0.cif


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

- Questions:
1. Does the procedure look reasonable to you? If something feels funky, what is it?
Yes, the procedure made entire sense but the original code kept on breaking when trying to install Boltz eventually creating an incorrect pointer to a nonsense MSA path. I also was wondering why you used "os.system" instead of ! or % to install the commands? Although, besides that, I believe I have a decent grasp on the code base. (Also interesting: the protein fold output seems to be different than the exemplar? Is this code base performed incorrectly? Or is there some new level of accuarcy with Boltz?)
2. Why does the MSA only have 1 line?
The MSA only has 1 line because it has no close detectable homologs to compare to in the database (as stated in your note). Thus, the MSA of this machine learning generated protein (de novo designed) draws blanks because nothing quite like it exists commonly in the real world naturally.
3. Does the results looking like something really interacting with ATP?
The ATP molecule appears to be separate from the protein, but does seem to sit in the middle of two coiled protein strands. A residue could potentially be interacting with the ATP molecule causing the protein to fold in a certain way (like how the residues closest to it are coiled, while the ones farther away are more uncoiled. Though, this could be due to other side-chain interactions).
4. Feel free to change the sequences to others to try again.
Also, are there specific things you would like me to find? Or would you like me to simply have fun with the science?



- Linna's Note --  *Actually, the sequence provided by the notebook is a de novo designed protein, aka it is generated by machine learning scripts.
De novo proteins are commonly less likely to be highly similar to native proteins, thus, when we are trying to build msa from this designed sequence, we will have a high chance to not be able to find any homologs, thus cannot build a real msa (since you cannot find other homolog sequences.).
However, as a de novo protein, you can see, the structure is simple, it is usually easily predictable by machine learning scripts (you actually didn't use a MSA to predict, most native proteins, you need to provide a real msa), highly stable, highly engineerable, thus significantly easier to make or improve in lab, and will significantly simplify the procedure to make a protein therapeutics.*